# 01 — Descarga y extracción de microdatos de mortalidad (INEGI)

**Proyecto:** chihuahua-suicide-seasonality-replication
**Etapa:** Diagnóstico inicial / preparación de datos crudos

Este notebook:
1. Descarga los archivos de microdatos de "Estadísticas de Defunciones Registradas" de INEGI, año por año (2008-2018).
2. Extrae los archivos comprimidos (.zip).
3. Carga cada año en un DataFrame de pandas (soporta CSV y DBF, ya que el formato ha cambiado entre años).
4. Estandariza nombres de columnas entre años (los catálogos de INEGI cambian ligeramente de un año a otro).
5. Concatena todo en un único dataset **sin filtrar por causa** (guardamos todas las causas de defunción; el filtro a suicidio, X60-X84, se hace en el notebook 02 de limpieza, de forma documentada).

⚠️ **Antes de correr este notebook necesitas completar `YEAR_URLS`** (celda 2) con el enlace directo de descarga de cada año. INEGI no usa una URL predecible por año (cada archivo tiene un ID interno distinto), así que hay que obtenerlas a mano:

1. Ve a https://www.inegi.org.mx/programas/edr/ → pestaña **Microdatos** (o el portal de "Descarga masiva": https://www.inegi.org.mx/app/descarga/default.html, tema = Mortalidad).
2. Para cada año 2008-2018, da clic derecho sobre el botón de descarga del archivo de datos → "Copiar enlace" (o "Copy link address").
3. Pega cada enlace en el diccionario `YEAR_URLS` de la celda siguiente.
4. Haz lo mismo para el diccionario de datos de cada año si quieres guardarlo también (opcional, pero recomendado para interpretar los códigos).

Este notebook está escrito para correr **en tu computadora** (no en este entorno de Claude), ya que aquí no tengo acceso de red a inegi.org.mx.

In [ ]:
import os
import io
import zipfile
import requests
import pandas as pd
from pathlib import Path

# --- Rutas del proyecto (ajusta si corres el notebook desde otra ubicación) ---
PROJECT_ROOT = Path("..")  # este notebook vive en notebooks/, el proyecto está un nivel arriba
RAW_DIR = PROJECT_ROOT / "data" / "raw" / "inegi_mortalidad"
RAW_ZIPS_DIR = RAW_DIR / "zips"
RAW_EXTRACTED_DIR = RAW_DIR / "extracted"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

for d in [RAW_ZIPS_DIR, RAW_EXTRACTED_DIR, INTERIM_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Carpetas listas:")
print(RAW_ZIPS_DIR)
print(RAW_EXTRACTED_DIR)
print(INTERIM_DIR)

In [ ]:
# --- TODO: completa con los enlaces reales de descarga por año ---
# Ve a https://www.inegi.org.mx/programas/edr/ (pestaña "Microdatos")
# o https://www.inegi.org.mx/app/descarga/default.html (tema = Mortalidad)
# y copia el enlace directo de descarga (normalmente termina en .zip) para cada año.

YEAR_URLS = {
    2008: "",  # <-- pega aquí el enlace de descarga de 2008
    2009: "",
    2010: "",
    2011: "",
    2012: "",
    2013: "",
    2014: "",
    2015: "",
    2016: "",
    2017: "",
    2018: "",
}

# Diccionarios de datos (codebooks) por año — opcional pero muy recomendado,
# ya que el microdato viene con códigos numéricos (sexo, entidad, causa CIE-10, etc.)
YEAR_DICTIONARY_URLS = {
    year: "" for year in YEAR_URLS
}

missing = [y for y, u in YEAR_URLS.items() if not u]
if missing:
    print(f"⚠️ Faltan URLs para los años: {missing}. Complétalas antes de continuar.")
else:
    print("✅ Todas las URLs de datos están completas.")

## 1. Descarga de archivos

In [ ]:
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024) -> Path:
    \"\"\"Descarga un archivo por streaming (evita cargarlo completo en memoria).\"\"\"
    if dest_path.exists():
        print(f"Ya existe, se omite descarga: {dest_path.name}")
        return dest_path

    print(f"Descargando: {url}")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    print(f"  -> Guardado en: {dest_path} ({dest_path.stat().st_size / 1e6:.1f} MB)")
    return dest_path


downloaded_zips = {}
for year, url in YEAR_URLS.items():
    if not url:
        print(f"[{year}] Sin URL, se omite.")
        continue
    dest = RAW_ZIPS_DIR / f"defunciones_{year}.zip"
    try:
        downloaded_zips[year] = download_file(url, dest)
    except Exception as e:
        print(f"[{year}] ERROR al descargar: {e}")

## 2. Extracción de los .zip

In [ ]:
def extract_zip(zip_path: Path, dest_dir: Path) -> list[Path]:
    \"\"\"Extrae un zip y regresa la lista de archivos extraídos.\"\"\"
    year_dir = dest_dir / zip_path.stem
    year_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(year_dir)
    extracted = list(year_dir.rglob("*"))
    extracted = [p for p in extracted if p.is_file()]
    return extracted


extracted_files_by_year = {}
for year, zip_path in downloaded_zips.items():
    try:
        files = extract_zip(zip_path, RAW_EXTRACTED_DIR)
        extracted_files_by_year[year] = files
        print(f"[{year}] Archivos extraídos:")
        for f in files:
            print(f"   - {f.name} ({f.suffix})")
    except Exception as e:
        print(f"[{year}] ERROR al extraer: {e}")

## 3. Carga de cada año a un DataFrame

INEGI ha cambiado el formato de entrega entre años (CSV en algunos, DBF en otros). Esta celda intenta detectar automáticamente el archivo de datos principal (excluye diccionarios/catálogos) y lo carga con el lector correcto.

**Nota:** si `dbfread` no está instalado y necesitas leer un `.dbf`, instala con `pip install dbfread --break-system-packages` (o `pip install simpledbf`).

In [ ]:
def find_main_data_file(files: list[Path]) -> Path | None:
    \"\"\"Heurística simple para encontrar el archivo de datos principal
    (evita catálogos/diccionarios, que suelen ser archivos más chicos o
    tener 'catalogo'/'diccionario' en el nombre).\"\"\"
    candidates = [
        f for f in files
        if f.suffix.lower() in (".csv", ".dbf")
        and "catalogo" not in f.name.lower()
        and "diccionario" not in f.name.lower()
        and "dic_" not in f.name.lower()
    ]
    if not candidates:
        return None
    # Nos quedamos con el archivo más grande (normalmente es el de microdatos)
    return max(candidates, key=lambda p: p.stat().st_size)


def load_year_dataframe(year: int, files: list[Path]) -> pd.DataFrame | None:
    main_file = find_main_data_file(files)
    if main_file is None:
        print(f"[{year}] No se encontró archivo de datos principal.")
        return None

    print(f"[{year}] Cargando: {main_file.name}")
    if main_file.suffix.lower() == ".csv":
        # INEGI a veces usa latin-1 en vez de utf-8
        try:
            df = pd.read_csv(main_file, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            df = pd.read_csv(main_file, encoding="latin-1", low_memory=False)
    elif main_file.suffix.lower() == ".dbf":
        from dbfread import DBF
        table = DBF(main_file, encoding="latin-1")
        df = pd.DataFrame(iter(table))
    else:
        print(f"[{year}] Formato no soportado: {main_file.suffix}")
        return None

    df["anio_archivo"] = year
    print(f"[{year}] Filas: {len(df):,} | Columnas: {len(df.columns)}")
    return df


yearly_dataframes = {}
for year, files in extracted_files_by_year.items():
    df_year = load_year_dataframe(year, files)
    if df_year is not None:
        yearly_dataframes[year] = df_year

## 4. Revisión de columnas por año (para detectar diferencias entre catálogos)

Antes de concatenar, revisamos qué tan parecidos son los nombres de columnas entre años. Es normal que haya diferencias (INEGI ha renombrado variables con el tiempo) — las documentaremos en `docs/METHODOLOGY.md` antes de estandarizar.

In [ ]:
for year, df_year in sorted(yearly_dataframes.items()):
    print(f"{year}: {list(df_year.columns)}")
    print()

## 5. Guardar el consolidado crudo (sin filtrar, sin estandarizar columnas todavía)

Guardamos cada año como está, más un intento de concatenación simple. La estandarización real de nombres de columnas y el filtrado a Chihuahua + CIE-10 X60-X84 se documentará y ejecutará en el siguiente notebook/script de limpieza (Etapa 2), una vez que revisemos juntas la salida de la celda anterior.

In [ ]:
# Guardamos cada año por separado (más seguro mientras los esquemas de columnas difieren)
for year, df_year in yearly_dataframes.items():
    out_path = INTERIM_DIR / f"defunciones_{year}_crudo.parquet"
    df_year.to_parquet(out_path, index=False)
    print(f"Guardado: {out_path}")

print()
print("Listo. Revisa la salida de la celda de columnas (sección 4) antes de continuar")
print("con la estandarización y el filtrado en el siguiente notebook.")